In [2]:
!nvidia-smi
!nvcc --version

Tue Apr 21 10:34:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%writefile hello.cu
#include <iostream>
__global__ void hello_world() {
  int tid = threadIdx.x + blockIdx.x * blockDim.x;
  printf("Hello, World! Thread %d\n", tid);
}
int main() {
  hello_world<<<1, 10>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing hello.cu


In [4]:
!nvcc -o hello hello.cu && ./hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello, World! Thread 0
Hello, World! Thread 1
Hello, World! Thread 2
Hello, World! Thread 3
Hello, World! Thread 4
Hello, World! Thread 5
Hello, World! Thread 6
Hello, World! Thread 7
Hello, World! Thread 8
Hello, World! Thread 9


In [5]:
%%writefile vector_add.cu
#include <iostream>
#include <cmath>

__global__ void vector_add(const float* A, float* B, float* C, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int N = 2;
    float A[10], B[10], C[10];

    for (int i = 0; i < N; ++i) {   // fix: ++i
        A[i] = (float)i + 1.0f;
        B[i] = 2.0;
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, N * sizeof(float));
    cudaMalloc(&d_b, N * sizeof(float));
    cudaMalloc(&d_c, N * sizeof(float));

    cudaMemcpy(d_a, A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, B, N * sizeof(float), cudaMemcpyHostToDevice);

    int blocksize = 256;
    int gridsize = (N + blocksize - 1) / blocksize;  // fix: gridsize can't be 0
    vector_add<<<gridsize, blocksize>>>(d_a, d_b, d_c, N);

    cudaMemcpy(C, d_c, N * sizeof(float), cudaMemcpyDeviceToHost);  // fix: copy result back

    printf("Vector A: ");
    for (int i = 0; i < N; ++i) std::cout << A[i] << ", ";  // fix: ++i

    printf("\nVector B: ");
    for (int i = 0; i < N; ++i) std::cout << B[i] << ", ";  // fix: ++i

    printf("\nVector C: ");
    for (int i = 0; i < N; ++i) std::cout << C[i] << ", ";  // fix: ++i

    printf("\n");

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing vector_add.cu


In [6]:
!nvcc -o vector_add vector_add.cu && ./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Vector A: 1, 2, 
Vector B: 2, 2, 
Vector C: 3, 4, 
